In [1]:
import sys
print(sys.executable)

C:\Users\durva\.anaconda\envs\rayoptics\python.exe


In [2]:
from rayoptics.environment import OpticalModel
from rayoptics.environment import Field
print("Import successful")

Import successful


In [3]:
opm = OpticalModel()
print("Model initialized")

import matplotlib.pyplot as plt
print("Matplotlib imported")

import matplotlib.pyplot as plt
import numpy as np


Model initialized
Matplotlib imported


In [10]:

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

np.seterr(all="ignore")          # NaN is used on purpose to flag failed rays

# ══ 1. Requirements ═══════════════════════════════════════════════════════════
# Sensor (ON Semi HAS3)
P_UM = 11.0
N_X, N_Y = 1280, 1280
P_MM = P_UM * 1e-3
HALF_DIAG_MM = 0.5 * float(np.hypot(N_X * P_MM, N_Y * P_MM))
HALF_SIDE_MM = 0.5 * min(N_X, N_Y) * P_MM

# Field definition. "inscribed": spec holds inside the circle touching the sensor sides,
# corners still imaged (lower weight). "diagonal": spec holds out to the corners.
FIELD_MODE = "inscribed"
EDGE_FRAC = HALF_SIDE_MM / HALF_DIAG_MM if FIELD_MODE == "inscribed" else 1.0   # of half-diagonal

# Sweep
EFL_LIST = [25, 30, 35, 40, 45, 50]
FNUM_LIST = [2.0, 2.4, 2.8, 3.2, 3.6, 4.0]
EFL_REF, FNUM_REF = 35, 2.8      # glass / stop choice is made here, then the sweep is warm-started
TOPOLOGIES = ["triplet", "doublet_singlet", "singlet_doublet"]   # 3-element designs (2-lens ones fail the spec)
# other options: "cemented", "spaced", "two_doublets", "five"

# Spot spec (geometric-mean sigma per axis, px).  MATLAB still makes the final call.
TARGET_SIGMA_PX = 1.0            # defocus drives the spec field toward this
SIGMA_MIN_PX = 0.5               # below this: pixel-locking risk
SIGMA_MAX_PX = 1.5               # spec ceiling (spec field, broadband, over temperature)
ELONG_MAX = 1.6                  # major/minor at the spec edge (reported flag)

# 1U packaging: 100 mm - 30 mm electronics - 6 mm screws - 4 mm front retainer/cover
L_MAX_MM = 100.0 - 30.0 - 6.0 - 4.0     # front-most of (stop, first vertex) -> image plane
# If True, the length limit is worked out for each (EFL, f/) from the 1U budget INCLUDING the baffle
# (same rule as the MATLAB check), so the optimiser only returns designs that fit:
#   L_max = 100 - PCB - fasteners - spare - baffle,  baffle = D / (tan(sun) - tan(half diag FOV)) + vane
LENGTH_FROM_1U_BUDGET = True
CUBE_MM, PCB_MM, FASTENER_MM, SPARE_MIN_MM = 100.0, 30.0, 6.0, 3.0
SUN_EXCL_DEG, VANE_MARGIN_MM = 45.0, 3.0
BFL_MIN_MM = 4.0                 # last vertex -> image: sensor window + mount clearance
MAX_SEMI_MM = 15.0               # max ray height on any surface (30 mm clear diameter)
FRONT_STOP_MAX_MM = 15.0         # how far in front of the first vertex a front stop may sit

# Thermal: +/-DT_K about nominal (40 K total), aluminium barrel / spacers
DT_K = 20.0
HOUSING_CTE = 23.6e-6
THERMAL_ALLOW_PX = 0.2           # allowed extra sigma from thermal defocus before penalty

# Transmission model: fraction of light kept per air-glass surface (AR-coated) and per lens
T_SURF, T_ELEM = 0.99, 0.99

# ══ 2. Glass catalogue (Schott, Sellmeier) ═════════════════════════════════════
# name: (B1, B2, B3, C1, C2, C3, nd, vd, dn/dT_abs [1e-6/K], CTE [1e-6/K])
# dn/dT and CTE from the SCHOTT datasheets: dn_abs/dT (in vacuum) at the e-line (546.1 nm) for
# +20/+40 degC, and CTE alpha for -30/+70 degC.
GLASS = {
    "N-BK7":   (1.03961212, 0.231792344, 1.01046945, 0.00600069867, 0.0200179144, 103.560653, 1.51680, 64.17, 1.6, 7.1),
    "N-K5":    (1.08511833, 0.199562005, 0.930511663, 0.00661099503, 0.024110866, 111.982777, 1.52249, 59.48, 0.7, 8.2),
    "N-BAK4":  (1.28834642, 0.132817724, 0.945395373, 0.00779980626, 0.0315631177, 105.965875, 1.56883, 55.98, 2.6, 7.0),
    "N-SK14":  (0.936155374, 0.594052018, 1.04374583, 0.00461716525, 0.016885927, 103.736265, 1.60311, 60.60, 1.7, 6.0),
    "N-SK16":  (1.34317774, 0.241144399, 0.994317969, 0.00704687339, 0.0229005, 92.7508526, 1.62041, 60.32, 0.9, 6.3),
    "N-LAK9":  (1.46231905, 0.344399589, 1.15508372, 0.00724270156, 0.0243353131, 85.4686868, 1.69100, 54.71, 2.2, 6.3),
    "N-LAK14": (1.50781212, 0.318866829, 1.14287213, 0.00746098727, 0.0242024834, 80.9565165, 1.69680, 55.41, 2.5, 5.5),
    "N-PK52A": (1.029607, 0.1880506, 0.736488165, 0.00516800155, 0.0166658798, 138.964129, 1.49700, 81.61, -7.7, 13.0),
    "N-FK51A": (0.971247817, 0.216901417, 0.904651666, 0.00472301995, 0.0153575612, 168.68133, 1.48656, 84.47, -7.0, 12.7),
    "N-BAF10": (1.5851495, 0.143559385, 1.08521269, 0.00926681282, 0.0424489805, 105.613573, 1.67003, 47.11, 3.5, 6.2),
    "N-F2":    (1.39757037, 0.159201403, 1.2686543, 0.00995906143, 0.0546931752, 119.248346, 1.62004, 36.37, 2.0, 7.8),
    "N-SF2":   (1.47343127, 0.163681849, 1.36920899, 0.0109019098, 0.0585683687, 127.404933, 1.64769, 33.82, 3.6, 6.7),
    "N-SF5":   (1.52481889, 0.187085527, 1.42729015, 0.011254756, 0.0588995392, 129.141675, 1.67270, 32.25, 2.0, 7.9),
    "N-SF10":  (1.62153902, 0.256287842, 1.64447552, 0.0122241457, 0.0595736775, 147.468793, 1.72828, 28.53, -0.1, 9.4),
    "N-SF6":   (1.77931763, 0.338149866, 2.08734474, 0.0133714182, 0.0617533621, 174.01759, 1.80518, 25.36, 0.0, 9.0),
    "N-SF11":  (1.73759695, 0.313747346, 1.89878101, 0.013188707, 0.0623068142, 155.23629, 1.78472, 25.68, 0.8, 8.5),
    "N-SF57":  (1.87543831, 0.37375749, 2.30001797, 0.0141749518, 0.0640509927, 177.389795, 1.84666, 23.78, 0.6, 8.5),
    "N-LASF9": (2.00029547, 0.298926886, 1.80691843, 0.0121426017, 0.0538736236, 156.530829, 1.85025, 32.17, 3.5, 7.4),
}


def n_glass(name, lam_nm):
    B1, B2, B3, C1, C2, C3 = GLASS[name][:6]
    l2 = (np.asarray(lam_nm, dtype=float) * 1e-3) ** 2
    return np.sqrt(1.0 + B1 * l2 / (l2 - C1) + B2 * l2 / (l2 - C2) + B3 * l2 / (l2 - C3))


# Check every entry against catalogue nd / vd; drop any with a coefficient typo.
for _g in list(GLASS):
    _nd = float(n_glass(_g, 587.56))
    _vd = (_nd - 1.0) / float(n_glass(_g, 486.13) - n_glass(_g, 656.27))
    if abs(_nd - GLASS[_g][6]) > 3e-4 or abs(_vd - GLASS[_g][7]) > 0.3:
        print(f"WARNING: dropping {_g} (Sellmeier check failed: nd={_nd:.5f}, vd={_vd:.2f})")
        del GLASS[_g]

DNDT = {g: v[8] * 1e-6 for g, v in GLASS.items()}
CTE = {g: v[9] * 1e-6 for g, v in GLASS.items()}

# Candidate glasses per element role. First entry = starting glass.
#   C = positive crown in a doublet, F = flint, P = positive singlet (high index helps Petzval)
ROLE_GLASSES = {
    "C": ["N-SK16", "N-BK7", "N-SK14", "N-BAK4", "N-LAK9", "N-PK52A", "N-FK51A"],
    "F": ["N-F2", "N-SF2", "N-SF5", "N-SF6", "N-SF10", "N-SF11", "N-BAF10"],
    "P": ["N-SK16", "N-LAK9", "N-LAK14", "N-SK14", "N-BAF10", "N-BK7", "N-LASF9"],
}
for _r in ROLE_GLASSES:
    ROLE_GLASSES[_r] = [g for g in ROLE_GLASSES[_r] if g in GLASS]

# ══ 3. Topologies ══════════════════════════════════════════════════════════════
# groups: elements in contact are cemented; groups are separated by air.
# stops : "front" = stop in air before the first lens; integer i = in the air gap after group i.
TOPO_DEF = {
    "cemented":        dict(groups=[["C", "F"]], stops=["front"]),
    "spaced":          dict(groups=[["C"], ["F"]], stops=["front", 0]),
    "doublet_singlet": dict(groups=[["C", "F"], ["P"]], stops=["front", 0]),
    "singlet_doublet": dict(groups=[["P"], ["C", "F"]], stops=["front", 0]),
    "triplet":         dict(groups=[["P"], ["F"], ["P"]], stops=[0, 1]),
    "two_doublets":    dict(groups=[["C", "F"], ["F", "C"]], stops=[0]),
    "five":            dict(groups=[["P"], ["C", "F"], ["P"], ["F"]], stops=[0, 1],
                            start=["N-SK16", "N-SK16", "N-SF6", "N-SK16", "N-F2"]),
}
THICK_BOUNDS = {"C": (1.5, 8.0), "P": (1.5, 8.0), "F": (1.0, 6.0)}
GAP_BOUNDS = (0.5, 20.0)
C_MAX = 4.0                      # |curvature| <= C_MAX / EFL
MIN_EDGE_GLASS, MIN_EDGE_AIR = 1.0, 0.5
STOP_CLEAR_MM = 0.5              # stop ring to glass, each side, at every radius the ring covers
# Buildability: every lens has one outer diameter = used aperture x SEMI_MARGIN + RIM_MM (seat).
# Neighbouring groups must keep MIN_GAP_MECH_MM apart across their FULL diameters, and the gap
# holding the stop must fit a machined ring STOP_THICK_MM thick plus STOP_CLEAR_MM on each side.
RIM_MM = 1.0
MIN_GAP_MECH_MM = 0.5
STOP_THICK_MM = 0.5
N_GAP_SAMPLES = 6
SEMI_MARGIN = 1.10

# ══ 4. Sampling ════════════════════════════════════════════════════════════════
# Design spectrum: star (~G type) x HAS3 response, rough.  EDIT if you have the QE curve.
WVL_NM = np.array([470.0, 550.0, 630.0, 720.0, 820.0])
WVL_WT = np.array([0.7, 1.0, 1.0, 0.75, 0.4])
REF_NM = 600.0
ND = {g: float(n_glass(g, REF_NM)) for g in GLASS}

FIELD_FRAC = np.unique(np.round([0.0, 0.4, EDGE_FRAC, 0.85, 1.0], 4))     # of half-diagonal
FIELD_W = np.where(FIELD_FRAC <= EDGE_FRAC + 1e-6, 1.0, 0.4)
FIELD_W[np.isclose(FIELD_FRAC, EDGE_FRAC)] = 1.5
I_EDGE = int(np.argmin(abs(FIELD_FRAC - EDGE_FRAC)))
SPEC_FIELDS = FIELD_FRAC <= EDGE_FRAC + 1e-6

VAL_FIELD_FRAC = np.unique(np.round([0.0, 0.2, 0.4, 0.55, EDGE_FRAC, 0.85, 1.0], 4))
VAL_WVL_NM = np.array([450.0, 500.0, 550.0, 600.0, 650.0, 700.0, 750.0, 800.0, 850.0])
VAL_WVL_WT = np.array([0.6, 0.85, 1.0, 1.0, 0.95, 0.85, 0.7, 0.5, 0.3])

# ══ 5. Optimiser effort ════════════════════════════════════════════════════════
EFFORT = dict(fast_rings=3, full_rings=6, val_rings=10,
              n_pool=2000, n_starts=16, nfev_start=250, n_polish=4, nfev_polish=150, n_basins=3, n_hops=40,
              sub_nfev=80, sub_passes=2, nfev_sweep=120, relax_passes=3)
SUB_AT_EACH_POINT = False        # True: re-run glass substitution at every (EFL, f/) (slow)
W_EFL, W_BFL, W_LEN, W_EDGE, W_CA, W_STOP, W_TH = 20.0, 20.0, 20.0, 50.0, 20.0, 50.0, 10.0
W_FAIL = 3.0                     # px-equivalent penalty per ray that misses a surface / hits TIR
SEED = 42


# ══ 6. Ray tracer ═════════════════════════════════════════════════════════════
def hexapolar(n_rings):
    px, py = [0.0], [0.0]
    for ring in range(1, n_rings + 1):
        rho, n_pts = ring / n_rings, 6 * ring
        a = 2.0 * np.pi * np.arange(n_pts) / n_pts
        px += list(rho * np.cos(a))
        py += list(rho * np.sin(a))
    return np.array(px), np.array(py)


def sag(c, h):
    rad = 1.0 - (c * h) ** 2
    return np.nan if rad <= 0.0 else c * h * h / (1.0 + np.sqrt(rad))


def sag_soft(c, h):
    """Sag at height h, plus how far h overshoots the hemisphere (|R|). Keeps the penalty smooth."""
    if abs(c) < 1e-12:
        return 0.0, 0.0
    hm = 0.999 / abs(c)
    return sag(c, min(h, hm)), max(0.0, h - hm)


def _hit(P, d, zv, c, allow_neg=False):
    if abs(c) < 1e-12:
        s = (zv - P[:, 2]) / d[:, 2]
    else:
        R = 1.0 / c
        Q = P - np.array([0.0, 0.0, zv + R])
        b = np.sum(d * Q, axis=1)
        disc = b * b - (np.sum(Q * Q, axis=1) - R * R)
        s = -b - np.sign(c) * np.sqrt(np.where(disc >= 0.0, disc, np.nan))
    if not allow_neg:
        s = np.where(s > 0.0, s, np.nan)
    X = P + s[:, None] * d
    if abs(c) < 1e-12:
        N = np.zeros_like(X)
        N[:, 2] = 1.0
    else:
        N = (X - np.array([0.0, 0.0, zv + R])) / abs(R)
    N = np.where((np.sum(N * d, axis=1) > 0.0)[:, None], -N, N)
    return X, N


def _refract(d, N, n_in, n_out):
    eta = np.asarray(n_in / n_out, dtype=float)
    if eta.ndim:
        eta = eta[:, None]
    cosi = -np.sum(d * N, axis=1)[:, None]
    k = 1.0 - eta ** 2 * (1.0 - cosi ** 2)
    return eta * d + (eta * cosi - np.sqrt(np.where(k >= 0.0, k, np.nan))) * N


def make_rays(half_diag_deg, r_ap, n_rings, field_frac, wvl_nm, wvl_wt, field_w):
    """Rays ordered (field, wavelength, pupil point); pupil coordinates normalised to r_ap."""
    px0, py0 = hexapolar(n_rings)
    N, F, W = px0.size, len(field_frac), len(wvl_nm)
    lam = np.tile(np.repeat(wvl_nm, N), F)
    wt = np.asarray(wvl_wt, float) / np.sum(wvl_wt)
    theta = np.arctan(np.tan(np.radians(half_diag_deg)) * np.asarray(field_frac))  # equal image-height steps
    return {"x": r_ap * np.tile(px0, F * W), "y": r_ap * np.tile(py0, F * W),
            "theta": np.repeat(theta, W * N), "theta_f": theta,
            "ng": {g: n_glass(g, lam) for g in GLASS},
            "N": N, "F": F, "W": W, "wts": wt[None, :, None, None], "w3": wt[None, :, None],
            "fw": np.asarray(field_w, float)[:, None, None, None]}


def trace(cs, zs, ga, rays, z0, ng=None):
    """Launch from the entrance-pupil plane z0 (relative to the first vertex).
    Rays that miss a surface or hit TIR become NaN; the mask says which survived."""
    ng = rays["ng"] if ng is None else ng
    x, y, th = rays["x"], rays["y"], rays["theta"]
    X = np.stack([x, y, np.full_like(x, z0)], axis=1)
    d = np.stack([np.zeros_like(x), np.sin(th), np.cos(th)], axis=1)
    n_prev, hmax = 1.0, np.empty(len(cs))
    for k, (c, z, g) in enumerate(zip(cs, zs, ga)):
        X, Nrm = _hit(X, d, z, c, allow_neg=(k == 0))
        h = np.hypot(X[:, 0], X[:, 1])
        if not np.isfinite(h).any():
            return None
        hmax[k] = float(np.nanmax(h))
        n_next = 1.0 if g is None else ng[g]
        d = _refract(d, Nrm, n_prev, n_next)
        n_prev = n_next
    ok = np.isfinite(X).all(axis=1) & np.isfinite(d).all(axis=1)
    if ok.mean() < 0.5:
        return None
    X, d = np.where(ok[:, None], X, 0.0), np.where(ok[:, None], d, np.array([0.0, 0.0, 1.0]))
    return X, d, hmax, ok


# ══ 7. Paraxial helpers ═══════════════════════════════════════════════════════
def paraxial(cs, zs, ga, nmap):
    """EFL and BFL (from last vertex)."""
    y, nu, n_prev = 1.0, 0.0, 1.0
    for k, (c, g) in enumerate(zip(cs, ga)):
        n_next = 1.0 if g is None else nmap[g]
        nu -= y * (n_next - n_prev) * c
        n_prev = n_next
        if k < len(cs) - 1:
            y += (zs[k + 1] - zs[k]) * nu / n_prev
    if abs(nu) < 1e-12:
        return np.nan, np.nan
    return -1.0 / nu, -y / nu


def para_y_at(zq, cs, zs, ga, nmap, h, u):
    """Height at plane zq of a paraxial ray entering with height h at the first vertex, slope u."""
    if zq <= zs[0]:
        return h + u * zq
    y, nu, n_prev = h, u, 1.0
    for k in range(len(cs)):
        n_next = 1.0 if ga[k] is None else nmap[ga[k]]
        nu -= y * (n_next - n_prev) * cs[k]
        n_prev = n_next
        zn = zs[k + 1] if k + 1 < len(cs) else np.inf
        if zq <= zn:
            return y + (zq - zs[k]) * nu / n_prev
        y += (zn - zs[k]) * nu / n_prev
    return np.nan


def pupil_from_stop(cs, zs, ga, nmap, z_stop, r_ap):
    """Entrance-pupil plane z_ep and the physical stop radius for entrance-pupil radius r_ap."""
    a = para_y_at(z_stop, cs, zs, ga, nmap, 1.0, 0.0)
    b = para_y_at(z_stop, cs, zs, ga, nmap, 0.0, 1.0)
    if not np.isfinite(a) or abs(a) < 1e-6:
        return np.nan, np.nan
    return b / a, abs(a) * r_ap


def count_air_glass(ga):
    prev_air, n = True, 0
    for g in ga:
        if (g is None) != prev_air:
            n += 1
        prev_air = g is None
    return n


# ══ 8. Topology bookkeeping ═══════════════════════════════════════════════════
def build_topo(name):
    D = TOPO_DEF[name]
    roles, ga_slot, tb, last_surf, first_surf = [], [], [], [], []
    for gi, grp in enumerate(D["groups"]):
        for role in grp:
            ga_slot.append(len(roles))
            roles.append(role)
            tb.append(THICK_BOUNDS[role])
        ga_slot.append(None)
        last_surf.append(len(ga_slot) - 1)
        first_surf.append(len(ga_slot) - 1 - len(grp))
        if gi < len(D["groups"]) - 1:
            tb.append(GAP_BOUNDS)
    start = D.get("start") or [ROLE_GLASSES[r][0] for r in roles]
    return dict(name=name, roles=roles, ga_slot=ga_slot, tb=tb, last_surf=last_surf, first_surf=first_surf,
                ngroups=len(D["groups"]),
                nsurf=len(ga_slot), nt=len(tb), n_lenses=len(roles), stops=D["stops"], start=list(start))


def bounds_for(T):
    lo = [-C_MAX] * T["nsurf"] + [b[0] for b in T["tb"]] + [0.0]
    hi = [C_MAX] * T["nsurf"] + [b[1] for b in T["tb"]] + [1.0]
    return np.array(lo), np.array(hi)


def decode(x, T, order, stop, efl):
    ns, nt = T["nsurf"], T["nt"]
    cs = np.asarray(x[:ns], float) / efl
    zs = np.concatenate([[0.0], np.cumsum(x[ns:ns + nt])])
    ga = [None if s is None else order[s] for s in T["ga_slot"]]
    p = float(x[-1])
    if stop == "front":
        z_stop = -p * FRONT_STOP_MAX_MM
    else:
        k = T["last_surf"][stop]
        z_stop = zs[k] + p * (zs[k + 1] - zs[k])
    return cs, zs, ga, z_stop


def stop_clearance(cs, zs, stop, T, z_stop, r_stop):
    """Axial clearance between the stop edge and the neighbouring lens surface(s) [mm]."""
    out = []
    if stop == "front":
        s0, e0 = sag_soft(cs[0], r_stop)
        out.append(zs[0] + s0 - z_stop - 5.0 * e0)
    else:
        k = T["last_surf"][stop]
        (s0, e0), (s1, e1) = sag_soft(cs[k], r_stop), sag_soft(cs[k + 1], r_stop)
        out.append(z_stop - (zs[k] + s0) - 5.0 * e0)
        out.append(zs[k + 1] + s1 - z_stop - 5.0 * e1)
    while len(out) < 2:
        out.append(10.0)
    return np.array(out)


def edge_slack(cs, zs, ga, hmax):
    out, need = np.empty(len(cs) - 1), np.empty(len(cs) - 1)
    for k in range(len(cs) - 1):
        h = SEMI_MARGIN * max(hmax[k], hmax[k + 1])
        (s0, e0), (s1, e1) = sag_soft(cs[k], h), sag_soft(cs[k + 1], h)
        out[k] = (zs[k + 1] + s1) - (zs[k] + s0) - 5.0 * (e0 + e1)
        need[k] = MIN_EDGE_AIR if ga[k] is None else MIN_EDGE_GLASS
    return out, need


def mech_gaps(cs, zs, T, hmax, stop, z_stop, r_stop):
    """Buildability check across full lens diameters.
    Returns (gap margins, stop margins) in mm; negative = rule broken.
    Each group of lenses gets one outer radius; beyond its own rim a surface is treated as a
    flat shoulder (the usual way a lens edge is finished), so deep curves still count."""
    G = T["ngroups"]
    R_out = [SEMI_MARGIN * max(hmax[T["first_surf"][g]:T["last_surf"][g] + 1]) + RIM_MM for g in range(G)]
    gap_m, stop_m = [], []
    for g in range(G - 1):
        kl, kr = T["last_surf"][g], T["first_surf"][g + 1]
        hs = np.linspace(0.0, max(R_out[g], R_out[g + 1]), N_GAP_SAMPLES)
        for h in hs:
            (sl, el), (sr, er) = sag_soft(cs[kl], min(h, R_out[g])), sag_soft(cs[kr], min(h, R_out[g + 1]))
            zl, zr = zs[kl] + sl, zs[kr] + sr
            gap_m.append(zr - zl - MIN_GAP_MECH_MM - 5.0 * (el + er))
            if stop == g:                      # the stop ring spans r_stop .. outer barrel
                if h >= r_stop - 1e-9 or h == hs[-1]:
                    half = 0.5 * STOP_THICK_MM
                    stop_m.append(min(z_stop - half - zl, zr - z_stop - half) - STOP_CLEAR_MM - 5.0 * (el + er))
                else:
                    stop_m.append(10.0)
    while len(stop_m) < N_GAP_SAMPLES:
        stop_m.append(10.0)
    while len(gap_m) < max(1, G - 1) * N_GAP_SAMPLES:
        gap_m.append(10.0)
    return np.array(gap_m), np.array(stop_m[:N_GAP_SAMPLES])


def thermal_model(cs, zs, ga, dT):
    """Geometry and index map at nominal + dT (glass grows with its CTE, air gaps with the barrel)."""
    cs2, t2 = np.array(cs, float), []
    for k in range(len(cs)):
        g = ga[k] if ga[k] is not None else (ga[k - 1] if k > 0 else None)
        a = CTE[g] if g is not None else HOUSING_CTE
        cs2[k] = cs[k] / (1.0 + a * dT)
        if k < len(cs) - 1:
            a_t = HOUSING_CTE if ga[k] is None else CTE[ga[k]]
            t2.append((zs[k + 1] - zs[k]) * (1.0 + a_t * dT))
    zs2 = np.concatenate([[0.0], np.cumsum(t2)])
    nmap = {g: ND[g] + DNDT[g] * dT for g in GLASS}
    return cs2, zs2, nmap


# ══ 9. Spot statistics ════════════════════════════════════════════════════════
def ray_weights(rays, ok):
    """Per-ray weight (F, W, N): wavelength weight, failed rays get 0, sums to 1 per field."""
    F, W, N = rays["F"], rays["W"], rays["N"]
    w = np.broadcast_to(rays["wts"][..., 0], (F, W, N)) * ok.reshape(F, W, N)
    s = w.sum(axis=(1, 2), keepdims=True)
    return w / np.where(s > 0, s, 1.0)


def spot_at(X, d, rays, z_img, w):
    F, W, N = rays["F"], rays["W"], rays["N"]
    sl = d[:, :2] / d[:, 2:3]
    a = (X[:, :2] + sl * (z_img - X[:, 2:3])).reshape(F, W, N, 2)
    cen = (w[..., None] * a).sum(axis=(1, 2))
    return a - cen[:, None, None, :], cen, sl.reshape(F, W, N, 2)


def best_focus_dz(X, d, rays, z_img, w):
    dd, _, s = spot_at(X, d, rays, z_img, w)
    fw = rays["fw"][..., 0]
    s = s - (w[..., None] * s).sum(axis=(1, 2))[:, None, None, :]
    den = (fw[..., None] * w[..., None] * s * s).sum()
    return -(fw[..., None] * w[..., None] * dd * s).sum() / den if den > 0 else 0.0


def spot_stats(dd, w):
    dx, dy = dd[..., 0], dd[..., 1]
    cxx, cyy, cxy = (w * dx * dx).sum(axis=(1, 2)), (w * dy * dy).sum(axis=(1, 2)), (w * dx * dy).sum(axis=(1, 2))
    tr, det = cxx + cyy, cxx * cyy - cxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4.0 - det, 0.0))
    k = 1.0 / P_MM
    return {"maj": np.sqrt(tr / 2.0 + disc) * k, "min": np.sqrt(np.maximum(tr / 2.0 - disc, 0.0)) * k,
            "geo": np.maximum(det, 0.0) ** 0.25 * k, "eq": np.sqrt(tr / 2.0) * k}


# ══ 10. Problem: residual vector for one (topology, glasses, stop, EFL, f/) ═══
def length_limit(efl, fno):
    """Allowed front-of-optics -> sensor length for this focal length and f-number [mm]."""
    if not LENGTH_FROM_1U_BUDGET:
        return L_MAX_MM
    half_diag = np.arctan(HALF_DIAG_MM / efl)
    baffle = (efl / fno) / (np.tan(np.radians(SUN_EXCL_DEG)) - np.tan(half_diag)) + VANE_MARGIN_MM
    return CUBE_MM - PCB_MM - FASTENER_MM - SPARE_MIN_MM - baffle


class Problem:
    def __init__(self, T, order, stop, efl, fno, rays):
        self.T, self.order, self.stop, self.efl, self.fno, self.rays = T, list(order), stop, efl, fno, rays
        self.r_ap = 0.5 * efl / fno
        self.L_max = length_limit(efl, fno)
        self.lo, self.hi = bounds_for(T)
        ns = T["nsurf"]
        self.n_extra = 4 + 2 + (ns - 1) + ns + max(1, T["ngroups"] - 1) * N_GAP_SAMPLES + N_GAP_SAMPLES
        self.n_rays = rays["F"] * rays["W"] * rays["N"]
        self.n_res = self.n_rays * 3 + self.n_extra

    def model(self, x):
        cs, zs, ga, z_stop = decode(x, self.T, self.order, self.stop, self.efl)
        efl, bfl = paraxial(cs, zs, ga, ND)
        if not (np.isfinite(efl) and efl > 0):
            return None
        z_ep, r_stop = pupil_from_stop(cs, zs, ga, ND, z_stop, self.r_ap)
        if not np.isfinite(z_ep) or abs(z_ep) > 200:
            return None
        out = trace(cs, zs, ga, self.rays, z_ep)
        if out is None:
            return None
        X, d, hmax, ok = out
        w = ray_weights(self.rays, ok)
        z_par = zs[-1] + bfl
        dz = best_focus_dz(X, d, self.rays, z_par, w)
        return dict(cs=cs, zs=zs, ga=ga, z_stop=z_stop, z_ep=z_ep, r_stop=r_stop, efl=efl, bfl=bfl,
                    X=X, d=d, hmax=hmax, ok=ok, w=w, z_best=z_par + dz)

    def thermal_px(self, m):
        """Extra spot sigma [px] from paraxial focus drift at +DT_K, sensor fixed to the barrel."""
        cs2, zs2, nm = thermal_model(m["cs"], m["zs"], m["ga"], DT_K)
        _, bfl2 = paraxial(cs2, zs2, m["ga"], nm)
        sensor = (m["z_best"] - m["zs"][-1]) * (1.0 + HOUSING_CTE * DT_K)
        delta = (bfl2 + (m["z_best"] - m["zs"][-1] - m["bfl"])) - sensor
        return abs(delta) / (4.0 * self.fno) / P_MM

    def residuals(self, x):
        m = self.model(x)
        if m is None:
            return np.full(self.n_res, 1e3)
        sl, need = edge_slack(m["cs"], m["zs"], m["ga"], m["hmax"])
        clr = stop_clearance(m["cs"], m["zs"], self.stop, self.T, m["z_stop"], m["r_stop"])
        if not (np.all(np.isfinite(sl)) and np.all(np.isfinite(clr))):
            return np.full(self.n_res, 1e3)
        R = self.rays
        dd, _, _ = spot_at(m["X"], m["d"], R, m["z_best"], m["w"])
        res = (np.sqrt(R["fw"] * m["w"][..., None]) * dd).ravel() / P_MM
        fail = W_FAIL * (~m["ok"]).astype(float)                # every lost ray costs W_FAIL px
        bfl_eff = m["z_best"] - m["zs"][-1]
        track = m["z_best"] - min(m["z_stop"], 0.0)
        extra = [W_EFL * (m["efl"] - self.efl), W_BFL * max(0.0, BFL_MIN_MM - bfl_eff),
                 W_LEN * max(0.0, track - self.L_max), W_TH * max(0.0, self.thermal_px(m) - THERMAL_ALLOW_PX)]
        extra += list(W_STOP * np.maximum(0.0, STOP_CLEAR_MM - clr))
        extra += list(W_EDGE * np.maximum(0.0, need - sl))
        gm, smg = mech_gaps(m["cs"], m["zs"], self.T, m["hmax"], self.stop, m["z_stop"], m["r_stop"])
        extra += list(W_EDGE * np.maximum(0.0, -gm)) + list(W_STOP * np.maximum(0.0, -smg))
        extra += list(W_CA * np.maximum(0.0, m["hmax"] - MAX_SEMI_MM))
        return np.concatenate([res, fail, extra])

    def cost(self, x):
        return float(np.sum(self.residuals(x) ** 2))

    def solve(self, x0, nfev):
        x0 = np.clip(x0, self.lo + 1e-9, self.hi - 1e-9)
        return least_squares(self.residuals, x0, bounds=(self.lo, self.hi), method="trf",
                             x_scale=np.maximum(self.hi - self.lo, 1e-3), max_nfev=nfev)


# ══ 11. Search strategies ═════════════════════════════════════════════════════
class RaySets:
    """Ray bundles for one (EFL, f/), shared by every topology/glass choice."""
    def __init__(self, efl, fno, effort):
        self.half_diag = float(np.degrees(np.arctan(HALF_DIAG_MM / efl)))
        r_ap = 0.5 * efl / fno
        self.fast = make_rays(self.half_diag, r_ap, effort["fast_rings"], FIELD_FRAC, WVL_NM, WVL_WT, FIELD_W)
        self.full = make_rays(self.half_diag, r_ap, effort["full_rings"], FIELD_FRAC, WVL_NM, WVL_WT, FIELD_W)
        self.val = make_rays(self.half_diag, r_ap, effort["val_rings"], VAL_FIELD_FRAC, VAL_WVL_NM, VAL_WVL_WT,
                             np.ones(len(VAL_FIELD_FRAC)))


def multistart(T, order, stop, efl, fno, rs, effort, rng):
    pf = Problem(T, order, stop, efl, fno, rs.fast)
    pool = rng.uniform(pf.lo, pf.hi, size=(effort["n_pool"], len(pf.lo)))
    cost = np.array([pf.cost(p) for p in pool])
    starts = pool[np.argsort(cost)[:effort["n_starts"]]]
    print(f"    [{T['name']}] stop={stop}: screened {len(pool)} random designs, optimising "
          f"{len(starts)} starts + {effort['n_hops']} hops ...", flush=True)
    runs = sorted((pf.solve(x0, effort["nfev_start"]) for x0 in starts), key=lambda r: r.cost)
    # basin hopping: random kicks (alternating small / large) from the best design, keep improvements.
    # The landscape is very multimodal; pure random starts find the good basin only ~1 time in 40.
    x_h, c_h = runs[0].x, runs[0].cost
    for i in range(effort["n_hops"]):
        kick = (0.25 if i % 2 else 0.08) * (pf.hi - pf.lo)
        r = pf.solve(np.clip(x_h + rng.normal(0.0, 1.0, len(x_h)) * kick, pf.lo, pf.hi), effort["nfev_start"])
        if r.cost < 0.99 * c_h:
            x_h, c_h = r.x, r.cost
            runs.insert(0, r)
    runs.sort(key=lambda r: r.cost)
    pF = Problem(T, order, stop, efl, fno, rs.full)
    pol = [pF.solve(r.x, effort["nfev_polish"]) for r in runs[:effort["n_polish"]]]
    return sorted(((r.cost, r.x) for r in pol), key=lambda t: t[0])      # [(cost, x), ...] best first


def glass_substitution(T, order, stop, x, efl, fno, rs, effort, log=True):
    """Coordinate descent over glasses: swap one element at a time, keep improvements."""
    order = list(order)
    p = Problem(T, order, stop, efl, fno, rs.fast)
    cur = p.solve(x, effort["sub_nfev"])
    x, c_cur = cur.x, cur.cost
    for _ in range(effort["sub_passes"]):
        changed = False
        for s, role in enumerate(T["roles"]):
            best = None
            for g in ROLE_GLASSES[role]:
                if g == order[s]:
                    continue
                trial = order[:s] + [g] + order[s + 1:]
                r = Problem(T, trial, stop, efl, fno, rs.fast).solve(x, effort["sub_nfev"])
                if best is None or r.cost < best[0]:
                    best = (r.cost, trial, r.x)
            if best is not None and best[0] < 0.98 * c_cur:
                if log:
                    print(f"      glass E{s + 1}: {order[s]} -> {best[1][s]}  cost {c_cur:.3g} -> {best[0]:.3g}",
                          flush=True)
                c_cur, order, x = best[0], best[1], best[2]
                changed = True
        if not changed:
            break
    pF = Problem(T, order, stop, efl, fno, rs.full)
    r = pF.solve(x, effort["nfev_polish"])
    return order, r.x, r.cost


def scale_x(x, T, efl_from, efl_to):
    x = np.array(x, float)
    ns, nt = T["nsurf"], T["nt"]
    x[ns:ns + nt] *= efl_to / efl_from
    return x


# ══ 12. Final evaluation: defocus to target, validation, thermal ══════════════
def defocus_to_target(m, rays, fno):
    """Pick the sensor plane that brings the spec-field sigma closest to TARGET_SIGMA_PX."""
    span = max(0.05, 6.0 * fno * TARGET_SIGMA_PX * P_MM)
    best = None
    w = np.where(SPEC_FIELDS, 1.0, 0.2) * np.where(np.arange(len(FIELD_FRAC)) == I_EDGE, 1.5, 1.0)
    for dz in np.linspace(-span, span, 241):
        st = spot_stats(spot_at(m["X"], m["d"], rays, m["z_best"] + dz, m["w"])[0], m["w"])
        geo, eq = st["geo"], st["eq"]
        c = np.sum(w * (eq - TARGET_SIGMA_PX) ** 2)                 # eq = rms/sqrt(2): sees both axes
        c += 20.0 * np.sum(np.maximum(0.0, geo[SPEC_FIELDS] - SIGMA_MAX_PX) ** 2)
        c += 5.0 * np.sum(np.maximum(0.0, SIGMA_MIN_PX - geo[SPEC_FIELDS]) ** 2)
        if best is None or c < best[0]:
            best = (c, dz)
    return best[1]


def thermal_spot(m, rays, z_img, dT):
    cs2, zs2, nm = thermal_model(m["cs"], m["zs"], m["ga"], dT)
    ng2 = {g: rays["ng"][g] + DNDT[g] * dT for g in GLASS}
    z_stop2 = m["z_stop"] * (1.0 + HOUSING_CTE * dT)
    z_ep, _ = pupil_from_stop(cs2, zs2, m["ga"], nm, z_stop2, 0.0)
    out = trace(cs2, zs2, m["ga"], rays, z_ep, ng=ng2)
    if out is None:
        return None
    sensor = zs2[-1] + (z_img - m["zs"][-1]) * (1.0 + HOUSING_CTE * dT)
    w = ray_weights(rays, out[3])
    return spot_stats(spot_at(out[0], out[1], rays, sensor, w)[0], w)


def finalise(T, order, stop, x, efl, fno, rs):
    p = Problem(T, order, stop, efl, fno, rs.full)
    m = p.model(x)
    if m is None:
        return None
    R = rs.full
    infocus = spot_stats(spot_at(m["X"], m["d"], R, m["z_best"], m["w"])[0], m["w"])
    dz = defocus_to_target(m, R, fno)
    z_img = m["z_best"] + dz
    dd, cen, _ = spot_at(m["X"], m["d"], R, z_img, m["w"])
    st = spot_stats(dd, m["w"])

    # broadband validation at the same sensor plane
    pv = Problem(T, order, stop, efl, fno, rs.val).model(x)
    if pv is None:
        sv = None
    else:
        sv = spot_stats(spot_at(pv["X"], pv["d"], rs.val, z_img, pv["w"])[0], pv["w"])
    v_spec = VAL_FIELD_FRAC <= EDGE_FRAC + 1e-6
    v_edge = int(np.argmin(abs(VAL_FIELD_FRAC - EDGE_FRAC)))

    # thermal (+/- DT_K) at the barrel-fixed sensor
    th = [thermal_spot(m, R, z_img, s * DT_K) for s in (-1.0, 1.0)]
    th_edge = max((t["geo"][I_EDGE] for t in th if t is not None), default=np.nan)
    th_cen = max((t["geo"][0] for t in th if t is not None), default=np.nan)

    sl, need = edge_slack(m["cs"], m["zs"], m["ga"], m["hmax"])
    clr = stop_clearance(m["cs"], m["zs"], stop, T, m["z_stop"], m["r_stop"])
    track = z_img - min(m["z_stop"], 0.0)
    bfl_eff = z_img - m["zs"][-1]
    rays_ok = bool(m["ok"].all() and (pv is None or pv["ok"].all()))
    gm, smg = mech_gaps(m["cs"], m["zs"], T, m["hmax"], stop, m["z_stop"], m["r_stop"])
    mech_ok = bool(np.all(gm >= -0.05) and np.all(smg >= -0.05))
    constraints_ok = bool(rays_ok and mech_ok and abs(m["efl"] - efl) < 0.05 and bfl_eff >= BFL_MIN_MM - 0.05 and track <= p.L_max + 0.05
                          and np.all(sl >= need - 0.05) and np.all(m["hmax"] <= MAX_SEMI_MM + 0.05)
                          and np.all(clr >= STOP_CLEAR_MM - 0.05))

    # distortion of the chief-ray centroid vs f*tan(theta)
    ideal = m["efl"] * np.tan(R["theta_f"])
    dist = np.where(ideal > 0, (cen[:, 1] - ideal) / np.where(ideal > 0, ideal, 1.0) * 100.0, 0.0)

    geo, maj, mnr = st["geo"], st["maj"], st["min"]
    elong = maj[I_EDGE] / mnr[I_EDGE] if mnr[I_EDGE] > 1e-9 else np.inf
    val_edge = sv["geo"][v_edge] if sv is not None else np.nan
    val_max = float(np.max(sv["geo"][v_spec])) if sv is not None else np.nan
    spec_vals = [*geo[SPEC_FIELDS], val_max, th_edge, th_cen]
    meets = bool(constraints_ok and np.all(np.isfinite(spec_vals)) and max(spec_vals) <= SIGMA_MAX_PX
                 and min(geo[SPEC_FIELDS]) >= SIGMA_MIN_PX)

    n_ag = count_air_glass(m["ga"])
    row = {
        "topology": T["name"], "n_lenses": T["n_lenses"], "glasses": "/".join(order), "EFL": efl, "f_number": fno,
        # --- read by MATLAB (same names as v1): geometric-mean sigma, major-axis sigma at the spec edge
        "sigma_center_px": round(geo[0], 3), "sigma_070_px": round(geo[I_EDGE], 3),
        "sigma_edge_px": round(geo[I_EDGE], 3), "sigma_edge_major_px": round(maj[I_EDGE], 3),
        "tau_est": round(T_SURF ** n_ag * T_ELEM ** T["n_lenses"], 4),
        "n_air_glass_surfaces": n_ag,
        "meets_spec": meets, "constraints_ok": constraints_ok, "elong_ok": bool(elong <= ELONG_MAX),
        # --- more spot detail
        "sigma_corner_px": round(geo[-1], 3), "sigma_edge_minor_px": round(mnr[I_EDGE], 3),
        "edge_elongation": round(elong, 2), "sigma_max_spec_px": round(float(geo[SPEC_FIELDS].max()), 3),
        "infocus_sigma_center_px": round(infocus["geo"][0], 3), "infocus_sigma_edge_px": round(infocus["geo"][I_EDGE], 3),
        "infocus_sigma_corner_px": round(infocus["geo"][-1], 3), "defocus_um": round(dz * 1e3, 1),
        "val_sigma_center_px": round(sv["geo"][0], 3) if sv else np.nan, "val_sigma_edge_px": round(val_edge, 3),
        "val_sigma_max_spec_px": round(val_max, 3),
        "thermal_sigma_center_px": round(th_cen, 3), "thermal_sigma_edge_px": round(th_edge, 3),
        **{f"sigma_f{int(round(f * 100)):03d}_px": round(g, 3) for f, g in zip(FIELD_FRAC, geo)},
        # --- first order / geometry
        "aperture_mm": round(2 * p.r_ap, 2), "half_fov_spec_deg": round(np.degrees(np.arctan(EDGE_FRAC * HALF_DIAG_MM / efl)), 2),
        "half_fov_diag_deg": round(rs.half_diag, 2), "plate_scale_arcsec_px": round(206264.8 * P_MM / m["efl"], 3),
        "efl_actual_mm": round(m["efl"], 3), "bfl_mm": round(bfl_eff, 2), "lens_stack_mm": round(m["zs"][-1], 2),
        "total_len_mm": round(track, 2), "length_limit_mm": round(p.L_max, 2), "tel_ratio": round(track / efl, 3), "min_edge_mm": round(float(sl.min()), 2),
        "min_gap_full_dia_mm": round(float(gm.min() + MIN_GAP_MECH_MM), 2),
        "stop_clear_min_mm": round(float(smg.min() + STOP_CLEAR_MM), 2), "mech_ok": mech_ok,
        "stop": str(stop), "stop_z_mm": round(m["z_stop"], 2), "stop_diam_mm": round(2 * m["r_stop"], 2),
        "ep_z_mm": round(m["z_ep"], 2), "distortion_corner_pct": round(float(dist[-1]), 3),
        "max_ray_height_mm": [round(float(h), 2) for h in m["hmax"]],
        "radii_mm": [round(float(1.0 / c), 4) if abs(c) > 1e-9 else float("inf") for c in m["cs"]],
        "thick_mm": [round(float(t), 4) for t in np.diff(m["zs"])],
        "cost": float(p.cost(x)), "x": [float(v) for v in np.round(x, 6)],
    }
    return row


def prescription(row):
    lines = [f"{row['topology']}  {row['glasses']}  EFL={row['EFL']} f/{row['f_number']}  "
             f"stop={row['stop']} at z={row['stop_z_mm']} mm (dia {row['stop_diam_mm']} mm)"]
    glasses = row["glasses"].split("/")
    T = build_topo(row["topology"])
    for k, (R, t) in enumerate(zip(row["radii_mm"], row["thick_mm"] + [row["bfl_mm"]])):
        s = T["ga_slot"][k]
        med = "air" if s is None else glasses[s]
        lines.append(f"  S{k + 1}: R={R:>10}  t={t:>7}  -> {med}")
    lines.append(f"  sigma (geo, px) centre/edge/corner = {row['sigma_center_px']}/{row['sigma_edge_px']}/"
                 f"{row['sigma_corner_px']}  defocus {row['defocus_um']} um  meets_spec={row['meets_spec']}")
    return "\n".join(lines)


# ══ 13. Driver ════════════════════════════════════════════════════════════════
def run_topology(name, efls, fnums, effort, ray_cache):
    T = build_topo(name)
    rng = np.random.default_rng(SEED)

    def rays(efl, fno):
        if (efl, fno) not in ray_cache:
            ray_cache[(efl, fno)] = RaySets(efl, fno, effort)
        return ray_cache[(efl, fno)]

    e_ref = min(efls, key=lambda e: abs(e - EFL_REF))
    f_ref = min(fnums, key=lambda f: abs(f - FNUM_REF))
    rs = rays(e_ref, f_ref)
    t0 = time.time()
    # 1. stop position & starting geometry at the reference point
    cands = []
    for stop in T["stops"]:
        res = multistart(T, T["start"], stop, e_ref, f_ref, rs, effort, rng)
        print(f"  [{name}] stop={stop}: best costs " + ", ".join(f"{c:.3g}" for c, _ in res), flush=True)
        cands += [(c, stop, x) for c, x in res]
    cands.sort(key=lambda t: t[0])
    # drop duplicates: several starts often converge to the same design (same stop, same x)
    lo, hi = bounds_for(T)
    uniq = []
    for c0, st, x0 in cands:
        if all(st != s2 or np.linalg.norm((x0 - x2) / (hi - lo)) > 0.02 for _, s2, x2 in uniq):
            uniq.append((c0, st, x0))
    cands = uniq
    # 2. glass substitution on the best few basins (a different basin often prefers different glass)
    best = None
    for c0, stop_i, x0 in cands[:effort["n_basins"]]:
        o_i, x_i, c_i = glass_substitution(T, T["start"], stop_i, x0, e_ref, f_ref, rs, effort)
        print(f"    basin cost {c0:.3g} stop={stop_i} -> {'/'.join(o_i)} cost {c_i:.3g}", flush=True)
        if best is None or c_i < best[0]:
            best = (c_i, stop_i, o_i, x_i)
    c, stop, order, x = best
    print(f"  [{name}] reference EFL={e_ref} f/{f_ref}: {'/'.join(order)} stop={stop} cost {c:.3g} "
          f"({time.time() - t0:.0f}s)", flush=True)

    # 3. sweep with warm starts from already-solved neighbours
    sol = {(e_ref, f_ref): (order, x, c)}
    grid = [(e, f) for e in efls for f in fnums]
    ie = {e: i for i, e in enumerate(efls)}
    jf = {f: j for j, f in enumerate(fnums)}
    todo = sorted(grid, key=lambda p: abs(ie[p[0]] - ie[e_ref]) + abs(jf[p[1]] - jf[f_ref]))

    def from_neighbours(e, f, keys):
        best = sol.get((e, f))
        best = None if best is None else (best[2], best[0], best[1])
        for k in keys:
            o_k, x_k, _ = sol[k]
            r = Problem(T, o_k, stop, e, f, rays(e, f).full).solve(scale_x(x_k, T, k[0], e), effort["nfev_sweep"])
            if best is None or r.cost < 0.995 * best[0]:
                best = (r.cost, o_k, r.x)
        return best

    t1 = time.time()
    for (e, f) in todo:                          # pass 1: grow outward from the reference
        if (e, f) in sol:
            continue
        nbr = [(abs(ie[e] - ie[k[0]]) + abs(jf[f] - jf[k[1]]), k) for k in sol]
        keys = [k for dist, k in sorted(nbr) if dist <= 2][:3] or [(e_ref, f_ref)]
        c, o, xb = from_neighbours(e, f, keys)
        if SUB_AT_EACH_POINT:
            o, xb, c = glass_substitution(T, o, stop, xb, e, f, rays(e, f), effort, log=False)
        sol[(e, f)] = (o, xb, c)
        print(f"    [{name}] solved EFL={e} f/{f}  cost {c:.3g}  ({len(sol)}/{len(grid)}, {time.time() - t1:.0f}s)",
              flush=True)
    for it in range(effort["relax_passes"]):     # pass 2+: let good basins spread to every neighbour
        n_imp = 0
        for (e, f) in todo:
            # 4-neighbours only, and only those whose design differs from this point's (others add nothing)
            keys = [k for k in sol if abs(ie[e] - ie[k[0]]) + abs(jf[f] - jf[k[1]]) == 1
                    and sol[k][2] < 0.9 * sol[(e, f)][2]]
            c_old = sol[(e, f)][2]
            c, o, xb = from_neighbours(e, f, keys)
            if c < 0.995 * c_old:
                sol[(e, f)] = (o, xb, c)
                n_imp += 1
        print(f"  [{name}] relaxation pass {it + 1}: {n_imp} points improved ({time.time() - t1:.0f}s)", flush=True)
        if n_imp == 0:
            break
    print(f"  [{name}] sweep done ({time.time() - t1:.0f}s)", flush=True)

    rows = []
    for (e, f) in sorted(grid):
        t1 = time.time()
        rs = rays(e, f)
        o, xb, c = sol[(e, f)]
        row = finalise(T, o, stop, xb, e, f, rs)
        if row is None:
            print(f"  [{name}] EFL={e} f/{f}: no valid design", flush=True)
            continue
        rows.append(row)
        print(f"  [{name}] EFL={e:>3} f/{f:<3} {row['glasses']:<28} sig c/e/corner="
              f"{row['sigma_center_px']:.2f}/{row['sigma_edge_px']:.2f}/{row['sigma_corner_px']:.2f} "
              f"(in-focus {row['infocus_sigma_center_px']:.2f}/{row['infocus_sigma_edge_px']:.2f}) "
              f"val={row['val_sigma_max_spec_px']:.2f} therm={row['thermal_sigma_edge_px']:.2f} "
              f"L={row['total_len_mm']} ok={row['constraints_ok']} SPEC={row['meets_spec']} "
              f"({time.time() - t1:.0f}s)", flush=True)
    return rows


def rank_designs(df):
    """Score every design that passes, pick the best per (EFL, f/) and overall.

    Rough figure of merit for cross-boresight accuracy (smaller = better), relative only:
      centroid error per star  ~ sigma / SNR,  SNR ~ D * sqrt(tau)   (photon-limited)
      angle per pixel ~ 1/EFL,  number of stars in the FOV ~ EFL^-2  -> 1/sqrt(N) ~ EFL
      => accuracy_index ~ sigma_max_spec / (D * sqrt(tau))     (EFL cancels)
    It ignores read noise, background and the magnitude-6 limit; MATLAB has the final say.
    """
    ok = df[df.meets_spec & df.constraints_ok].copy()
    if not len(ok):
        return ok, ok
    ok["accuracy_index"] = ok.sigma_max_spec_px / (ok.aperture_mm * np.sqrt(ok.tau_est))
    ok["accuracy_index"] /= ok.accuracy_index.min()          # 1.00 = best design found
    ok = ok.sort_values(["n_lenses", "accuracy_index"])
    per_point = ok.drop_duplicates(["EFL", "f_number"]).sort_values(["EFL", "f_number"])
    overall = ok.sort_values(["accuracy_index", "n_lenses"])
    return per_point, overall


def main(topos=None, efls=None, fnums=None, effort=None, prefix="star_tracker", resume=True):
    topos, efls, fnums = topos or TOPOLOGIES, efls or EFL_LIST, fnums or FNUM_LIST
    effort = effort or EFFORT
    print(f"Running topologies {topos} | EFL {efls} | f/ {fnums}", flush=True)
    print(f"Sensor half-diagonal {HALF_DIAG_MM:.2f} mm, spec field = {EDGE_FRAC:.3f} of it ({FIELD_MODE}); "
          f"length limit: {'1U budget incl. baffle' if LENGTH_FROM_1U_BUDGET else str(L_MAX_MM) + ' mm'}; glasses: {len(GLASS)}", flush=True)
    ray_cache, frames = {}, []
    for name in topos:
        csv = f"{prefix}_{name}_optics.csv"
        if resume and os.path.exists(csv):
            print(f"\n=== {name} === already done, loading {csv} (delete it to re-run)", flush=True)
            frames.append(pd.read_csv(csv))
            continue
        print(f"\n=== {name} ===", flush=True)
        rows = run_topology(name, efls, fnums, effort, ray_cache)
        if not rows:
            continue
        df = pd.DataFrame(rows).sort_values(["EFL", "f_number"])
        df.to_csv(csv, index=False)                             # saved per topology -> safe to stop & resume
        with open(f"{prefix}_{name}_prescription.txt", "w") as fh:
            fh.write("\n\n".join(prescription(r) for r in df.to_dict("records")))
        frames.append(df)
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(f"{prefix}_all_runs.csv", index=False)

    print("\nFewest lenses meeting the spot spec per (EFL, f/):")
    tab = df.assign(n=np.where(df.meets_spec & df.constraints_ok, df.n_lenses, 99)
                    ).groupby(["EFL", "f_number"]).n.min().unstack()
    print(tab.replace(99, "-").to_string())

    per_point, overall = rank_designs(df)
    per_point.to_csv(f"{prefix}_summary.csv", index=False)
    overall.to_csv(f"{prefix}_ranked.csv", index=False)
    if len(overall):
        cols = ["topology", "n_lenses", "glasses", "EFL", "f_number", "aperture_mm", "sigma_center_px",
                "sigma_edge_px", "sigma_corner_px", "val_sigma_max_spec_px", "thermal_sigma_edge_px",
                "total_len_mm", "tau_est", "accuracy_index"]
        print("\nBest designs that pass (accuracy_index 1.00 = best; lower is better):")
        print(overall[cols].head(15).to_string(index=False))
        fewest = overall[overall.n_lenses == overall.n_lenses.min()].iloc[0]
        best = overall.iloc[0]
        print(f"\nBest overall      : {best.topology} ({best.n_lenses} lenses) {best.glasses} "
              f"EFL={best.EFL} f/{best.f_number}  accuracy_index={best.accuracy_index:.2f}")
        print(f"Best, fewest lens : {fewest.topology} ({fewest.n_lenses} lenses) {fewest.glasses} "
              f"EFL={fewest.EFL} f/{fewest.f_number}  accuracy_index={fewest.accuracy_index:.2f}")
    else:
        print("\nNo design meets the spec. Look at sigma_max_spec_px in the CSVs for the closest ones.")
    return df


if __name__ == "__main__":
    # python star_tracker_opt.py [test|quick|full|all] [topology ...]
    #   all   = every topology (2 to 5 lenses), full grid, then ranks them
    #   e.g.  python star_tracker_opt.py quick triplet
    #         python star_tracker_opt.py full triplet doublet_singlet
    mode = sys.argv[1] if len(sys.argv) > 1 else "full"
    topos = [t for t in sys.argv[2:] if t in TOPO_DEF] or (list(TOPO_DEF) if mode == "all" else TOPOLOGIES)
    bad = [t for t in sys.argv[2:] if t not in TOPO_DEF]
    if bad:
        print(f"unknown topology {bad}; choose from {list(TOPO_DEF)}")
    if mode == "test":
        eff = dict(EFFORT, n_pool=300, n_starts=4, nfev_start=150, n_polish=2, nfev_polish=80, n_basins=1, n_hops=4,
                   sub_nfev=40, sub_passes=1, full_rings=4, val_rings=6, relax_passes=1)
        main(topos, [35], [2.8], eff, prefix="star_tracker_test")
    elif mode == "quick":
        eff = dict(EFFORT, n_pool=800, n_starts=8, nfev_start=200, n_polish=3, nfev_polish=100, n_basins=2, n_hops=15,
                   sub_nfev=60, sub_passes=2, full_rings=5, val_rings=8, nfev_sweep=80, relax_passes=2)
        main(topos, [30, 35, 40, 45, 50], [2.4, 2.8, 3.2, 4.0], eff, prefix="star_tracker_quick")
    else:
        main(topos)


unknown topology ['C:\\Users\\durva\\AppData\\Roaming\\jupyter\\runtime\\kernel-8e06aad4-9024-4ee8-af45-cb29bb326d08.json']; choose from ['cemented', 'spaced', 'doublet_singlet', 'singlet_doublet', 'triplet', 'two_doublets', 'five']
Running topologies ['triplet', 'doublet_singlet', 'singlet_doublet'] | EFL [25, 30, 35, 40, 45, 50] | f/ [2.0, 2.4, 2.8, 3.2, 3.6, 4.0]
Sensor half-diagonal 9.96 mm, spec field = 0.707 of it (inscribed); length limit: 1U budget incl. baffle; glasses: 18

=== triplet ===
    [triplet] stop=0: screened 2000 random designs, optimising 16 starts + 40 hops ...
  [triplet] stop=0: best costs 41.9, 42.5, 42.6, 42.6
    [triplet] stop=1: screened 2000 random designs, optimising 16 starts + 40 hops ...
  [triplet] stop=1: best costs 13, 13, 13, 13
      glass E1: N-SK16 -> N-LAK14  cost 12.6 -> 11.3
      glass E2: N-F2 -> N-SF5  cost 11.3 -> 7.84
      glass E3: N-SK16 -> N-LAK14  cost 7.84 -> 6.34
    basin cost 13 stop=1 -> N-LAK14/N-SF5/N-LAK14 cost 6.91
      g

In [11]:
import pandas as pd, os

f = "star_tracker_doublet_singlet_optics.csv"
if not os.path.exists(f):
    f = "final_doublet_singlet_optics.csv"      # in case you used prefix="final"

r = pd.read_csv(f)
r = r[(r.EFL == 40) & (r.f_number == 3.6)].iloc[0]
for c in ["glasses", "radii_mm", "thick_mm", "bfl_mm", "lens_stack_mm", "total_len_mm",
          "stop", "stop_z_mm", "stop_diam_mm", "max_ray_height_mm",
          "min_gap_full_dia_mm", "stop_clear_min_mm"]:
    print(c, "=", r[c])

glasses = N-BAK4/N-F2/N-LASF9
radii_mm = [14.4232, -10.8834, 12.4986, 31.7103, -111.7272]
thick_mm = [8.0, 5.2073, 1.9296, 2.0123]
bfl_mm = 26.05
lens_stack_mm = 17.15
total_len_mm = 43.2
stop = front
stop_z_mm = -0.0
stop_diam_mm = 11.11
max_ray_height_mm = [5.87, 5.8, 5.84, 6.27, 6.4]
min_gap_full_dia_mm = 0.5
stop_clear_min_mm = 10.5


In [12]:

import numpy as np
import pandas as pd
from rayoptics.environment import OpticalModel, PupilSpec, FieldSpec, WvlSpec
from rayoptics.raytr.trace import trace_base

# ── which design to check ───────────────────────────────────────────────────
CSV = "star_tracker_doublet_singlet_optics.csv"   # the selected design
EFL, FNUM = 40, 3.6

# ── must match the optimiser settings ───────────────────────────────────────
P_MM = 0.011
HALF_DIAG_MM = 0.5 * np.hypot(1280 * P_MM, 1280 * P_MM)
WVL_NM = [470.0, 550.0, 630.0, 720.0, 820.0]
WVL_WT = [0.7, 1.0, 1.0, 0.75, 0.4]
FIELD_FRAC = [0.0, 0.4, 0.7071, 0.85, 1.0]     # fraction of the way to the sensor corner
N_RINGS = 6

# which lens (index into the glass list) follows each surface; None = air
SLOTS = {"triplet":         [0, None, 1, None, 2, None],
         "singlet_doublet": [0, None, 1, 2, None],
         "doublet_singlet": [0, 1, None, 2, None]}
# last surface of each lens group (the stop "0" sits in the air gap after group 0)
LAST_SURF = {"triplet": [1, 3, 5], "singlet_doublet": [1, 4], "doublet_singlet": [2, 4]}


def parse_list(s):
    s = str(s).replace("np.float64(", "").replace(")", "").strip("[] ")
    return [float(v) for v in s.split(",")]


def hexapolar(n_rings):
    px, py = [0.0], [0.0]
    for ring in range(1, n_rings + 1):
        a = 2 * np.pi * np.arange(6 * ring) / (6 * ring)
        px += list(ring / n_rings * np.cos(a))
        py += list(ring / n_rings * np.sin(a))
    return np.array(px), np.array(py)


def flat(r):                     # RayOptics uses radius 0 for a flat surface
    return 0.0 if not np.isfinite(r) else r


# ── 1. read the design ──────────────────────────────────────────────────────
df = pd.read_csv(CSV)
row = df[(df.EFL == EFL) & (df.f_number == FNUM)].iloc[0]
# Use the full-precision design vector, not radii_mm / thick_mm (rounded to 0.01 mm,
# which is enough to move the focus by tens of microns).
slots = SLOTS[row.topology]
ns = len(slots)
x = parse_list(row.x)
R = [EFL / c if abs(c) > 1e-12 else float("inf") for c in x[:ns]]   # x holds curvature * EFL
t = x[ns:ns + ns - 1]
glasses = row.glasses.split("/")
stop, z_stop, bfl = str(row.stop), float(row.stop_z_mm), float(row.bfl_mm)
print(f"Checking {row.topology} {row.glasses}  EFL={EFL} f/{FNUM}  stop={stop} at z={z_stop} mm")

# ── 2. build it in RayOptics ────────────────────────────────────────────────
opm = OpticalModel()
sm, osp = opm["seq_model"], opm["optical_spec"]
opm.radius_mode = True
osp["pupil"] = PupilSpec(osp, key=["object", "epd"], value=EFL / FNUM)
half_diag = np.degrees(np.arctan(HALF_DIAG_MM / EFL))
angles = [float(np.degrees(np.arctan(np.tan(np.radians(half_diag)) * f))) for f in FIELD_FRAC]
osp["fov"] = FieldSpec(osp, key=["object", "angle"], flds=angles)
osp["wvls"] = WvlSpec(list(zip(WVL_NM, WVL_WT)), ref_wl=2)
sm.gaps[0].thi = 1e10            # star at infinity

if stop == "front":              # flat stop plane in front of the first lens
    sm.add_surface([0.0, -z_stop])
    sm.set_stop()
stop_after = None if stop == "front" else LAST_SURF[row.topology][int(stop)]
z_vertex = np.concatenate([[0.0], np.cumsum(t)])
for k, r in enumerate(R):
    gap = t[k] if k < len(t) else bfl                    # last gap = lens to sensor
    g = slots[k]
    if k == stop_after:          # split this air gap: lens surface -> stop plane -> next lens
        d1 = z_stop - z_vertex[k]
        sm.add_surface([flat(r), d1])
        sm.add_surface([0.0, gap - d1])
        sm.set_stop()
    elif g is None:
        sm.add_surface([flat(r), gap])
    else:
        sm.add_surface([flat(r), gap, glasses[g], "Schott"])
opm.update_model()

# ── 3. focal length ─────────────────────────────────────────────────────────
fod = opm["analysis_results"]["parax_data"].fod
print(f"\nFocal length:  ours {row.efl_actual_mm:.3f} mm   RayOptics {fod.efl:.3f} mm")

# ── 4. trace the same pupil grid and colours as ours; keep landing point + slope ──
px, py = hexapolar(N_RINGS)
w_wl = np.array(WVL_WT) / np.sum(WVL_WT)
bundles = []                                  # per field: points (n,2), slopes (n,2), weights (n)
for fi, fld in enumerate(osp["fov"].fields):
    pts, slp, wts = [], [], []
    for wl, ww in zip(WVL_NM, w_wl):
        for xx, yy in zip(px, py):
            try:
                ray = trace_base(opm, [xx, yy], fld, wl)[0]
                p_img, d_img = np.asarray(ray[-1][0]), np.asarray(ray[-2][1])
                pts.append(p_img[:2]); slp.append(d_img[:2] / d_img[2]); wts.append(ww)
            except Exception:
                pass                                       # ray missed a surface
    W = np.array(wts)
    bundles.append((np.array(pts), np.array(slp), W / W.sum()))


def sigmas(dz):
    """Geometric-mean sigma [px] per field with the sensor moved by dz [mm]."""
    out = []
    for P, S, W in bundles:
        Q = P + S * dz
        d = Q - (W[:, None] * Q).sum(axis=0)
        cxx, cyy, cxy = (W * d[:, 0] ** 2).sum(), (W * d[:, 1] ** 2).sum(), (W * d[:, 0] * d[:, 1]).sum()
        out.append(max(cxx * cyy - cxy ** 2, 0.0) ** 0.25 / P_MM)
    return np.array(out)


# ── 5a. same sensor position as ours ────────────────────────────────────────
ours = [row.get(f"sigma_f{int(round(f * 100)):03d}_px", np.nan) for f in FIELD_FRAC]
sig_ro = sigmas(0.0)
print("\nA) Sensor at our position     ours   RayOptics   difference")
for f, a, b in zip(FIELD_FRAC, ours, sig_ro):
    print(f"  field {f:4.2f}                  {a:5.2f}     {b:5.2f}      {100 * (b - a) / a:+5.1f} %")

# ── 5b. best focus in each program (does not depend on tiny focus differences) ──
FW = np.array([1.0, 1.0, 1.5, 0.4, 0.4])                  # same field weights as the optimiser
dzs = np.linspace(-0.3, 0.3, 601)
rms = [np.sum(FW * sigmas(dz) ** 2) for dz in dzs]
dz_best = dzs[int(np.argmin(rms))]
best = sigmas(dz_best)
ours_inf = [row.infocus_sigma_center_px, row.infocus_sigma_edge_px, row.infocus_sigma_corner_px]
idx = [0, 2, 4]
print(f"\nB) At best focus (RayOptics best focus is {dz_best * 1e3:+.0f} um from our sensor plane, "
      f"ours is {-row.defocus_um:+.0f} um)")
print("                              ours   RayOptics   difference")
for name, a, i in zip(["centre", "edge", "corner"], ours_inf, idx):
    print(f"  {name:6s}                      {a:5.2f}     {best[i]:5.2f}      {100 * (best[i] - a) / a:+5.1f} %")

Checking doublet_singlet N-BAK4/N-F2/N-LASF9  EFL=40 f/3.6  stop=front at z=-0.0 mm

Focal length:  ours 39.999 mm   RayOptics 40.023 mm

A) Sensor at our position     ours   RayOptics   difference
  field 0.00                   1.01      1.03       +1.8 %
  field 0.40                   0.61      0.62       +1.7 %
  field 0.71                   0.80      0.78       -1.5 %
  field 0.85                   1.02      1.02       -0.7 %
  field 1.00                   1.84      1.83       -0.2 %

B) At best focus (RayOptics best focus is +10 um from our sensor plane, ours is +6 um)
                              ours   RayOptics   difference
  centre                       0.98      0.97       -0.7 %
  edge                         0.82      0.83       +0.7 %
  corner                       1.84      1.85       +0.1 %
